<a href="https://colab.research.google.com/github/KanitAon/ebay-soccer-card-market-analysis/blob/main/src/data_analysis/How_Relation_Between_Player_Performance_and_Card_Sell_Price.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Football Card Market + Player Performance

It is designed to run directly in **Google Colab** using only three input datasets:

1. eBay football-card market data (`.csv`, `.xlsx`, or `.xls`)
2. player match statistics (`.csv`, `.xlsx`, or `.xls`)
3. player profiles (`.csv`, `.xlsx`, or `.xls`)

### What was cleaned

- Removed duplicated imports, plots, and repeated calculations.
- Removed hard-coded local/project paths.
- Removed dependencies on intermediate Parquet files.
- Rebuilt player-role mapping directly from `Player_Profiles`.
- Standardized eBay price, brand, boolean, player-ID, and product-line fields.
- Added one output directory for all figures and tables.
- Organized the workflow into Q1–Q5 sections so `Runtime → Run all` works from top to bottom.

In [ ]:
# ============================================================
# 0. SETUP
# ============================================================

from pathlib import Path
import sys
import warnings
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from scipy.stats import (
    spearmanr,
    mannwhitneyu,
    kruskal,
    pearsonr,
)
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# One place for all generated files.
OUTPUT_DIR = Path("football_card_analysis_output")
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Shared project palette.
COLOR_BLUE = "#2B6CB0"
COLOR_GREEN = "#2E7D32"
COLOR_YELLOW = "#F2C94C"
COLOR_RED = "#D71921"
COLOR_GRAY = "#718096"
COLOR_DARK = "#1A202C"

print("Output directory:", OUTPUT_DIR.resolve())

## 1. Upload the three input datasets once

Run the next cell and upload all three input files together in **one upload action**.

Supported formats can be mixed:

- `.csv`
- `.xlsx`
- `.xls`

The notebook identifies each dataset automatically from its column names, so filenames do **not** need to follow a fixed naming convention.

In [ ]:
# ============================================================
# 1. LOAD INPUT FILES ONCE (.csv, .xlsx, or .xls)
# ============================================================

SUPPORTED_EXTENSIONS = {".csv", ".xlsx", ".xls"}

if "google.colab" in sys.modules:
    from google.colab import files

    # Upload all 3 datasets together in ONE upload action.
    uploaded = files.upload()

    input_paths = [
        Path(name)
        for name in uploaded.keys()
        if Path(name).suffix.lower() in SUPPORTED_EXTENSIONS
    ]
else:
    # Portable fallback for Jupyter outside Colab.
    input_paths = [
        path
        for path in Path.cwd().iterdir()
        if path.is_file()
        and path.suffix.lower() in SUPPORTED_EXTENSIONS
    ]

if len(input_paths) < 3:
    raise FileNotFoundError(
        "Please upload all 3 input datasets together. "
        "Supported formats: .csv, .xlsx, .xls"
    )


def read_input_file(path, nrows=None):
    """
    Read CSV or Excel using one shared function.
    """
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(
            path,
            nrows=nrows,
            low_memory=False,
        )

    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(
            path,
            nrows=nrows,
        )

    raise ValueError(
        f"Unsupported file format: {suffix}"
    )


def identify_input_file(path):
    """
    Identify dataset type from its columns rather than filename.
    """
    sample = read_input_file(
        path,
        nrows=3,
    )

    columns = set(sample.columns)

    if {
        "asking_price_usd",
        "Player_Canonical",
        "brand",
        "product_line",
        "FotMob_Player_ID",
    }.issubset(columns):
        return "ebay"

    if {
        "Season",
        "Match_ID",
        "Player_ID",
        "Minutes",
        "Rating",
        "Team",
    }.issubset(columns):
        return "matches"

    if {
        "FotMob_Player_ID",
        "Player_Canonical",
        "Final_Position",
        "Primary_Position",
    }.issubset(columns):
        return "profiles"

    return None


dataset_paths = {}

for path in input_paths:
    dataset_type = identify_input_file(path)

    if dataset_type is not None:
        if dataset_type in dataset_paths:
            raise ValueError(
                f"More than one file was identified as '{dataset_type}'. "
                f"Please upload only one file for each required dataset."
            )

        dataset_paths[dataset_type] = path


required = {
    "ebay",
    "matches",
    "profiles",
}

missing = required - set(dataset_paths)

if missing:
    raise ValueError(
        "Could not identify all required datasets. "
        f"Missing: {sorted(missing)}\n\n"
        "Required datasets:\n"
        "1. eBay Market Data\n"
        "2. Player Match Statistics\n"
        "3. Player Profiles"
    )


print("Detected input files:")
for key, path in dataset_paths.items():
    print(
        f"  {key:8s}: {path.name} "
        f"({path.suffix.lower()})"
    )


# Read each dataset only once after identification.
ebay = read_input_file(
    dataset_paths["ebay"]
)

matches = read_input_file(
    dataset_paths["matches"]
)

profiles = read_input_file(
    dataset_paths["profiles"]
)


print("\nRaw shapes:")
print(
    "  eBay listings  :",
    ebay.shape,
)

print(
    "  Match records  :",
    matches.shape,
)

print(
    "  Player profiles:",
    profiles.shape,
)

In [ ]:
# ============================================================
# 2. CLEAN AND STANDARDIZE INPUT DATA
# ============================================================

def clean_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    return (
        series.astype("string")
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "yes": True,
            "no": False,
        })
        .fillna(False)
        .astype(bool)
    )


# ---------- eBay ----------
numeric_ebay = [
    "asking_price_usd",
    "serial",
    "grade",
    "year",
    "FotMob_Player_ID",
    "Opta_ID",
]

for col in numeric_ebay:
    if col in ebay.columns:
        ebay[col] = pd.to_numeric(ebay[col], errors="coerce")

ebay["FotMob_Player_ID"] = ebay["FotMob_Player_ID"].astype("Int64")
ebay["Opta_ID"] = ebay["Opta_ID"].astype("Int64")
ebay["year"] = ebay["year"].astype("Int64")

for col in ["graded", "autograph", "patch", "rookie", "is_auction"]:
    if col in ebay.columns:
        ebay[col] = clean_bool(ebay[col])

ebay["brand_clean"] = (
    ebay["brand"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "panini": "Panini",
        "topps": "Topps",
    })
)

ebay["product_line"] = (
    ebay["product_line"]
    .astype("string")
    .str.strip()
    .replace({
        "": pd.NA,
        "nan": pd.NA,
        "<NA>": pd.NA,
        "None": pd.NA,
    })
)

# Keep valid positive asking-price records.
ebay = ebay.loc[
    ebay["asking_price_usd"].notna()
    & (ebay["asking_price_usd"] > 0)
].copy()

ebay["price_log"] = np.log1p(ebay["asking_price_usd"])


# ---------- Match statistics ----------
matches["Player_ID"] = pd.to_numeric(
    matches["Player_ID"], errors="coerce"
)

matches["Opta_ID"] = (
    matches["Opta_ID"]
    .astype("string")
    .str.strip()
    .replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "<NA>": pd.NA,
    })
)

matches["Minutes"] = pd.to_numeric(
    matches["Minutes"], errors="coerce"
).fillna(0)

matches["Rating"] = pd.to_numeric(
    matches["Rating"], errors="coerce"
)


# ---------- Player profiles ----------
profiles["FotMob_Player_ID"] = pd.to_numeric(
    profiles["FotMob_Player_ID"], errors="coerce"
).astype("Int64")

profiles["Opta_ID"] = pd.to_numeric(
    profiles["Opta_ID"], errors="coerce"
).astype("Int64")


print("=" * 72)
print("CLEAN DATA VALIDATION")
print("=" * 72)

print(f"eBay listings       : {len(ebay):,}")
print(f"Unique eBay players : {ebay['FotMob_Player_ID'].nunique():,}")
print(f"Match rows          : {len(matches):,}")
print(f"Player profiles     : {len(profiles):,}")
print(f"Missing eBay prices : {ebay['asking_price_usd'].isna().sum():,}")

display(
    ebay[
        [
            "Player_Canonical",
            "asking_price_usd",
            "brand_clean",
            "product_line",
            "graded",
            "autograph",
            "patch",
            "rookie",
        ]
    ].head()
)

# Q1 — What Does the Football Card Asking-Price Market Look Like?

This section combines the market overview, listing concentration, card-characteristic analysis, and the useful player/team views from the original notebooks.

In [ ]:
# ============================================================
# Q1.1 — MARKET OVERVIEW
# ============================================================

market_overview = pd.DataFrame({
    "Metric": [
        "Listings",
        "Unique players",
        "Unique sellers",
        "Brands",
        "Product lines",
        "Median asking price (USD)",
        "Mean asking price (USD)",
    ],
    "Value": [
        len(ebay),
        ebay["FotMob_Player_ID"].nunique(),
        ebay["seller"].nunique(),
        ebay["brand_clean"].nunique(),
        ebay["product_line"].nunique(dropna=True),
        ebay["asking_price_usd"].median(),
        ebay["asking_price_usd"].mean(),
    ],
})

display(market_overview)

market_overview.to_csv(
    TABLE_DIR / "q1_market_overview.csv",
    index=False
)

In [ ]:
# ============================================================
# Q1.2 — ASKING-PRICE DISTRIBUTION
# ============================================================

price_bins = [
    0,
    25,
    50,
    100,
    250,
    500,
    1000,
    np.inf,
]

price_labels = [
    "< $25",
    "$25–49",
    "$50–99",
    "$100–249",
    "$250–499",
    "$500–999",
    "$1,000+",
]

ebay["price_band"] = pd.cut(
    ebay["asking_price_usd"],
    bins=price_bins,
    labels=price_labels,
    right=False,
)

price_distribution = (
    ebay["price_band"]
    .value_counts(sort=False)
    .rename("Listings")
    .to_frame()
)

price_distribution["Share (%)"] = (
    price_distribution["Listings"]
    / len(ebay)
    * 100
)

display(price_distribution.round(2))

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)

bars = ax.bar(
    price_distribution.index.astype(str),
    price_distribution["Share (%)"],
)

for bar, value in zip(
    bars,
    price_distribution["Share (%)"],
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.4,
        f"{value:.1f}%",
        ha="center",
        va="bottom",
        fontsize=9,
    )

ax.set_title(
    "Distribution of Seller Asking Prices",
    fontsize=16,
    fontweight="bold",
)
ax.set_xlabel("Asking-price band")
ax.set_ylabel("Share of listings (%)")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.20)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "q1_price_distribution.png",
    dpi=500,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# ============================================================
# Q1.3 — LISTING CONCENTRATION ACROSS PLAYERS
# ============================================================

player_listings = (
    ebay
    .groupby(
        ["FotMob_Player_ID", "Player_Canonical"],
        dropna=False,
        as_index=False,
    )
    .agg(
        Listings=("source_row_id", "nunique"),
        Median_Asking_Price_USD=("asking_price_usd", "median"),
        Mean_Asking_Price_USD=("asking_price_usd", "mean"),
        Min_Asking_Price_USD=("asking_price_usd", "min"),
        Max_Asking_Price_USD=("asking_price_usd", "max"),
    )
    .sort_values("Listings", ascending=False)
    .reset_index(drop=True)
)

player_listings["Cumulative_Listings"] = (
    player_listings["Listings"].cumsum()
)

player_listings["Cumulative_Share_Pct"] = (
    player_listings["Cumulative_Listings"]
    / player_listings["Listings"].sum()
    * 100
)

players_for_80 = int(
    (player_listings["Cumulative_Share_Pct"] < 80).sum() + 1
)

print(
    f"Players needed to reach at least 80% of all listings: "
    f"{players_for_80:,} of {len(player_listings):,}"
)

display(player_listings.head(20))

player_listings.to_csv(
    TABLE_DIR / "q1_player_listing_summary.csv",
    index=False,
)


fig, ax = plt.subplots(figsize=(10, 6), dpi=150)

x = np.arange(1, len(player_listings) + 1)

ax.plot(
    x,
    player_listings["Cumulative_Share_Pct"],
    linewidth=2,
)

ax.axhline(
    80,
    linestyle="--",
    linewidth=1,
    alpha=0.65,
)

ax.axvline(
    players_for_80,
    linestyle="--",
    linewidth=1,
    alpha=0.65,
)

ax.set_title(
    "Cumulative Concentration of eBay Listings Across Players",
    fontsize=15,
    fontweight="bold",
)
ax.set_xlabel("Number of players, ranked by listing count")
ax.set_ylabel("Cumulative share of listings (%)")
ax.grid(alpha=0.20)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "q1_player_listing_concentration.png",
    dpi=500,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# ============================================================
# Q1.4 — CARD CHARACTERISTICS AND ASKING PRICE
# ============================================================

feature_rows = []

for feature in ["autograph", "patch", "rookie"]:
    no_group = ebay.loc[
        ebay[feature] == False,
        "asking_price_usd",
    ].dropna()

    yes_group = ebay.loc[
        ebay[feature] == True,
        "asking_price_usd",
    ].dropna()

    u_stat, p_value = mannwhitneyu(
        yes_group,
        no_group,
        alternative="two-sided",
    )

    feature_rows.append({
        "Feature": feature.title(),
        "Feature_n": len(yes_group),
        "Non_Feature_n": len(no_group),
        "Feature_Median_USD": yes_group.median(),
        "Non_Feature_Median_USD": no_group.median(),
        "Feature_Mean_USD": yes_group.mean(),
        "Non_Feature_Mean_USD": no_group.mean(),
        "Median_Ratio": (
            yes_group.median() / no_group.median()
            if no_group.median() != 0
            else np.nan
        ),
        "Mann_Whitney_U": u_stat,
        "P_Value": p_value,
    })

feature_tests = pd.DataFrame(feature_rows)

display(feature_tests.round(4))

feature_tests.to_csv(
    TABLE_DIR / "q1_feature_price_tests.csv",
    index=False,
)


# ---------- Serial-number rarity ----------
serial_cards = ebay.loc[
    ebay["serial"].notna()
    & (ebay["serial"] > 0)
].copy()

serial_cards["Serial_Rarity_Group"] = pd.cut(
    serial_cards["serial"],
    bins=[0, 10, 25, 50, 99, 199, np.inf],
    labels=[
        "/1–10",
        "/11–25",
        "/26–50",
        "/51–99",
        "/100–199",
        "/200+",
    ],
)

serial_summary = (
    serial_cards
    .groupby(
        "Serial_Rarity_Group",
        observed=False,
    )["asking_price_usd"]
    .agg(
        Listings="count",
        Median_USD="median",
        Mean_USD="mean",
    )
    .reset_index()
)

serial_rho, serial_p = spearmanr(
    serial_cards["serial"],
    serial_cards["asking_price_usd"],
    nan_policy="omit",
)

print(
    f"Serial print run vs asking price: "
    f"Spearman rho = {serial_rho:.3f}, "
    f"p = {serial_p:.3g}"
)

display(serial_summary.round(2))

serial_summary.to_csv(
    TABLE_DIR / "q1_serial_rarity_summary.csv",
    index=False,
)


fig, ax = plt.subplots(figsize=(10, 6), dpi=150)

bars = ax.bar(
    serial_summary["Serial_Rarity_Group"].astype(str),
    serial_summary["Median_USD"],
)

for bar, median_price, n in zip(
    bars,
    serial_summary["Median_USD"],
    serial_summary["Listings"],
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() * 1.02,
        f"${median_price:,.0f}\n(n={n})",
        ha="center",
        va="bottom",
        fontsize=9,
    )

ax.set_title(
    "Median Asking Price by Serial-Number Rarity",
    fontsize=15,
    fontweight="bold",
)
ax.set_xlabel("Printed serial-number range")
ax.set_ylabel("Median asking price (USD)")
ax.grid(axis="y", alpha=0.20)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "q1_serial_rarity_price.png",
    dpi=500,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# ============================================================
# Q1.5 — GRADING
# ============================================================

grading_summary = (
    ebay
    .groupby("graded")["asking_price_usd"]
    .agg(
        Listings="count",
        Median_USD="median",
        Mean_USD="mean",
    )
    .rename(index={
        False: "Raw / ungraded",
        True: "Graded",
    })
)

display(grading_summary.round(2))

graded_price = ebay.loc[
    ebay["graded"] == True,
    "asking_price_usd",
].dropna()

raw_price = ebay.loc[
    ebay["graded"] == False,
    "asking_price_usd",
].dropna()

grading_u, grading_p = mannwhitneyu(
    graded_price,
    raw_price,
    alternative="two-sided",
)

print(
    f"Graded vs raw Mann–Whitney p-value: "
    f"{grading_p:.4g}"
)


grade_9 = ebay.loc[
    ebay["grade"] == 9,
    "asking_price_usd",
].dropna()

grade_10 = ebay.loc[
    ebay["grade"] == 10,
    "asking_price_usd",
].dropna()

grade_u, grade_p = mannwhitneyu(
    grade_10,
    grade_9,
    alternative="two-sided",
)

grade_9_10_summary = pd.DataFrame({
    "Grade": [9, 10],
    "Listings": [len(grade_9), len(grade_10)],
    "Median_USD": [grade_9.median(), grade_10.median()],
    "Mean_USD": [grade_9.mean(), grade_10.mean()],
})

display(grade_9_10_summary.round(2))

print(
    f"Grade 10 vs Grade 9 Mann–Whitney p-value: "
    f"{grade_p:.4g}"
)

grading_summary.to_csv(
    TABLE_DIR / "q1_grading_summary.csv"
)

grade_9_10_summary.to_csv(
    TABLE_DIR / "q1_grade_9_vs_10.csv",
    index=False,
)


fig, ax = plt.subplots(figsize=(8, 6), dpi=150)

plot_grade = grading_summary.reset_index()
bars = ax.bar(
    plot_grade["graded"],
    plot_grade["Median_USD"],
)

for bar, value, n in zip(
    bars,
    plot_grade["Median_USD"],
    plot_grade["Listings"],
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() * 1.02,
        f"${value:,.0f}\n(n={n})",
        ha="center",
        va="bottom",
        fontsize=10,
    )

ax.set_title(
    "Median Asking Price: Graded vs Raw Cards",
    fontsize=15,
    fontweight="bold",
)
ax.set_xlabel("")
ax.set_ylabel("Median asking price (USD)")
ax.grid(axis="y", alpha=0.20)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "q1_graded_vs_raw_price.png",
    dpi=500,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# ============================================================
# Q1.6 — BRAND AND PRODUCT-LINE COMPARISON
# ============================================================

brand_summary = (
    ebay
    .groupby("brand_clean")["asking_price_usd"]
    .agg(
        Listings="count",
        Median_USD="median",
        Mean_USD="mean",
    )
    .sort_values("Listings", ascending=False)
)

display(brand_summary.round(2))

panini_price = ebay.loc[
    ebay["brand_clean"] == "Panini",
    "asking_price_usd",
].dropna()

topps_price = ebay.loc[
    ebay["brand_clean"] == "Topps",
    "asking_price_usd",
].dropna()

brand_u, brand_p = mannwhitneyu(
    panini_price,
    topps_price,
    alternative="two-sided",
)

print(
    f"Panini vs Topps Mann–Whitney p-value: "
    f"{brand_p:.4g}"
)


product_line_summary = (
    ebay
    .dropna(subset=["brand_clean", "product_line"])
    .groupby(
        ["brand_clean", "product_line"],
        as_index=False,
    )
    .agg(
        Listings=("source_row_id", "nunique"),
        Median_USD=("asking_price_usd", "median"),
        Mean_USD=("asking_price_usd", "mean"),
    )
    .sort_values(
        ["brand_clean", "Listings"],
        ascending=[True, False],
    )
)

display(product_line_summary.head(30).round(2))

brand_summary.to_csv(
    TABLE_DIR / "q1_brand_summary.csv"
)

product_line_summary.to_csv(
    TABLE_DIR / "q1_product_line_summary.csv",
    index=False,
)


# Kruskal–Wallis test across sufficiently populated product lines
MIN_PRODUCT_LINE_N = 50
kruskal_rows = []

for brand in ["Panini", "Topps"]:
    brand_data = ebay.loc[
        ebay["brand_clean"] == brand
    ].copy()

    line_counts = brand_data["product_line"].value_counts()
    valid_lines = line_counts[
        line_counts >= MIN_PRODUCT_LINE_N
    ].index

    groups = [
        brand_data.loc[
            brand_data["product_line"] == line,
            "asking_price_usd",
        ].dropna()
        for line in valid_lines
    ]

    if len(groups) >= 2:
        h_stat, p_value = kruskal(*groups)

        kruskal_rows.append({
            "Brand": brand,
            "Product_Lines_Tested": len(groups),
            "Kruskal_H": h_stat,
            "P_Value": p_value,
        })

kruskal_results = pd.DataFrame(kruskal_rows)
display(kruskal_results.round(4))

kruskal_results.to_csv(
    TABLE_DIR / "q1_product_line_kruskal_tests.csv",
    index=False,
)

In [ ]:
# ============================================================
# Q1.7 — SELECTED PANINI vs TOPPS PRODUCT TIERS
# Adapted from new_big_picture.ipynb
# ============================================================

tier_mapping = {
    "Select": "Mid",
    "Finest": "Mid",
    "Prizm": "Mid-High",
    "Chrome": "Mid-High",
    "Immaculate": "Luxury",
    "Dynasty": "Luxury",
}

tier_order = [
    "Mid",
    "Mid-High",
    "Luxury",
]

tier_data = ebay.loc[
    ebay["product_line"].isin(tier_mapping)
].copy()

tier_data["Tier"] = (
    tier_data["product_line"]
    .map(tier_mapping)
)

tier_summary = (
    tier_data
    .groupby(
        ["Tier", "brand_clean", "product_line"],
        as_index=False,
    )
    .agg(
        Listings=("source_row_id", "nunique"),
        Mean_USD=("asking_price_usd", "mean"),
        Median_USD=("asking_price_usd", "median"),
    )
)

display(
    tier_summary
    .sort_values(["Tier", "brand_clean"])
    .round(2)
)

tier_summary.to_csv(
    TABLE_DIR / "q1_selected_tier_summary.csv",
    index=False,
)


fig, ax = plt.subplots(figsize=(10, 6), dpi=150)

x = np.arange(len(tier_order))
width = 0.36

panini_values = []
topps_values = []

for tier in tier_order:
    panini_row = tier_summary.loc[
        (tier_summary["Tier"] == tier)
        & (tier_summary["brand_clean"] == "Panini"),
        "Mean_USD",
    ]

    topps_row = tier_summary.loc[
        (tier_summary["Tier"] == tier)
        & (tier_summary["brand_clean"] == "Topps"),
        "Mean_USD",
    ]

    panini_values.append(
        panini_row.iloc[0] if len(panini_row) else np.nan
    )
    topps_values.append(
        topps_row.iloc[0] if len(topps_row) else np.nan
    )

ax.bar(
    x - width / 2,
    panini_values,
    width,
    label="Panini",
    color=COLOR_YELLOW,
)

ax.bar(
    x + width / 2,
    topps_values,
    width,
    label="Topps",
    color=COLOR_RED,
)

ax.set_title(
    "Average Card Asking Price by Selected Product Tier",
    fontsize=15,
    fontweight="bold",
)
ax.set_xlabel("Selected product tier")
ax.set_ylabel("Average asking price (USD)")
ax.set_xticks(x)
ax.set_xticklabels(tier_order)
ax.legend()
ax.grid(axis="y", alpha=0.20)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "q1_selected_tier_average_price.png",
    dpi=500,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# ============================================================
# Q1.8 — PLAYER-LEVEL MARKET VIEWS
# ============================================================

top_supply = (
    player_listings
    .nlargest(30, "Listings")
    .sort_values("Listings")
)

fig, ax = plt.subplots(figsize=(12, 10), dpi=150)

bars = ax.barh(
    top_supply["Player_Canonical"],
    top_supply["Listings"],
)

for bar, value in zip(
    bars,
    top_supply["Listings"],
):
    ax.text(
        bar.get_width() + 2,
        bar.get_y() + bar.get_height() / 2,
        f"{int(value):,}",
        va="center",
        fontsize=9,
    )

ax.set_title(
    "Top 30 Players by Number of eBay Card Listings",
    fontsize=15,
    fontweight="bold",
)
ax.set_xlabel("Observed listings")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.20)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "q1_top30_players_by_listing_count.png",
    dpi=500,
    bbox_inches="tight",
)
plt.show()


top_median_price = (
    player_listings
    .nlargest(30, "Median_Asking_Price_USD")
    .sort_values("Median_Asking_Price_USD")
)

fig, ax = plt.subplots(figsize=(12, 10), dpi=150)

bars = ax.barh(
    top_median_price["Player_Canonical"],
    top_median_price["Median_Asking_Price_USD"],
)

for bar, price, count in zip(
    bars,
    top_median_price["Median_Asking_Price_USD"],
    top_median_price["Listings"],
):
    ax.text(
        bar.get_width() * 1.02,
        bar.get_y() + bar.get_height() / 2,
        f"${price:,.0f} ({count} cards)",
        va="center",
        fontsize=8.5,
    )

ax.set_title(
    "Top 30 Players by Median Card Asking Price",
    fontsize=15,
    fontweight="bold",
)
ax.set_xlabel("Median asking price (USD)")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.20)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "q1_top30_players_by_median_price.png",
    dpi=500,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# ============================================================
# Q1.9 — TEAM MARKET VIEW
# Adapted from new_big_picture.ipynb
# ============================================================

# Use the most frequently observed FotMob team for each Opta ID.
match_team = matches.copy()

match_team["Opta_ID_Clean"] = (
    match_team["Opta_ID"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
)

ebay["Opta_ID_Clean"] = (
    ebay["Opta_ID"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
)

team_mapping = (
    match_team
    .dropna(subset=["Opta_ID_Clean", "Team"])
    .groupby("Opta_ID_Clean")["Team"]
    .agg(
        lambda x:
        x.mode().iloc[0]
        if not x.mode().empty
        else pd.NA
    )
)

ebay["Team"] = ebay["Opta_ID_Clean"].map(
    team_mapping
)

team_market = (
    ebay
    .dropna(subset=["Team"])
    .groupby("Team", as_index=False)
    .agg(
        Listings=("source_row_id", "nunique"),
        Median_Asking_Price_USD=("asking_price_usd", "median"),
        Mean_Asking_Price_USD=("asking_price_usd", "mean"),
    )
    .sort_values("Listings", ascending=False)
)

display(team_market.head(20).round(2))

team_market.to_csv(
    TABLE_DIR / "q1_team_market_summary.csv",
    index=False,
)


top_team_supply = (
    team_market
    .head(20)
    .sort_values("Listings")
)

fig, ax = plt.subplots(figsize=(11, 8), dpi=150)

bars = ax.barh(
    top_team_supply["Team"],
    top_team_supply["Listings"],
)

for bar, value in zip(
    bars,
    top_team_supply["Listings"],
):
    ax.text(
        bar.get_width() + 2,
        bar.get_y() + bar.get_height() / 2,
        f"{int(value):,}",
        va="center",
        fontsize=9,
    )

ax.set_title(
    "Top Teams by Observed Card Listing Volume",
    fontsize=15,
    fontweight="bold",
)
ax.set_xlabel("Observed listings")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.20)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "q1_team_listing_volume.png",
    dpi=500,
    bbox_inches="tight",
)
plt.show()

# Q2 — Player Performance

The original player-performance notebook depended on intermediate role and Parquet files.  
This version reconstructs the same role framework from `Player_Profiles.csv` and aggregates `Player_Match_Statistics.csv` directly.

In [ ]:
# ============================================================
# Q2.1 — BUILD ANALYSIS ROLES FROM PLAYER PROFILES
# ============================================================

role_map = {
    "Keeper": "Goalkeeper",
    "Center Back": "Centre-back",

    "Left Back": "Full-back / Wing-back",
    "Right Back": "Full-back / Wing-back",
    "Left Wing-Back": "Full-back / Wing-back",
    "Right Wing-Back": "Full-back / Wing-back",

    "Defensive Midfielder": "Central / Defensive Midfielder",
    "Central Midfielder": "Central / Defensive Midfielder",
    "MID": "Central / Defensive Midfielder",

    "Attacking Midfielder": "Attacking Midfielder",

    "Left Winger": "Wide Mid. / Winger",
    "Right Winger": "Wide Mid. / Winger",
    "Left Midfielder": "Wide Mid. / Winger",
    "Right Midfielder": "Wide Mid. / Winger",

    "Striker": "Striker",
}

roles = profiles.copy()

roles["Player_Key"] = (
    "FM_"
    + roles["FotMob_Player_ID"].astype("string")
)

roles["Player"] = roles["Player_Canonical"]

roles["Analysis_Role"] = (
    roles["Final_Position"]
    .map(role_map)
)

if roles["Analysis_Role"].isna().any():
    display(
        roles.loc[
            roles["Analysis_Role"].isna(),
            [
                "Player_Canonical",
                "Final_Position",
            ],
        ]
    )
    raise ValueError(
        "Some Final_Position values are not mapped to an Analysis_Role."
    )

role_order = [
    "Goalkeeper",
    "Centre-back",
    "Full-back / Wing-back",
    "Central / Defensive Midfielder",
    "Attacking Midfielder",
    "Wide Mid. / Winger",
    "Striker",
]

print(
    "Role records:",
    f"{len(roles):,}"
)

display(
    roles[
        [
            "Player_Canonical",
            "Final_Position",
            "Analysis_Role",
        ]
    ].head(10)
)

In [ ]:
# ============================================================
# Q2.2 — BUILD ELIGIBLE PLAYER-SEASON PERFORMANCE UNIVERSE
# ============================================================

df = matches.copy()

# ---------- Canonical player key ----------
valid_fotmob = df["Player_ID"].gt(0)

crosswalk = (
    df.loc[
        valid_fotmob
        & df["Opta_ID"].notna(),
        ["Opta_ID", "Player_ID"],
    ]
    .drop_duplicates()
)

crosswalk_check = (
    crosswalk
    .groupby("Opta_ID")["Player_ID"]
    .nunique()
)

if len(crosswalk_check):
    assert (
        crosswalk_check.max() <= 1
    ), "One Opta ID maps to multiple FotMob IDs."

opta_to_fotmob = (
    crosswalk
    .drop_duplicates("Opta_ID")
    .set_index("Opta_ID")["Player_ID"]
)

df["Resolved_Player_ID"] = df["Player_ID"]

missing_fotmob = ~df["Resolved_Player_ID"].gt(0)

df.loc[
    missing_fotmob,
    "Resolved_Player_ID",
] = (
    df.loc[
        missing_fotmob,
        "Opta_ID",
    ]
    .map(opta_to_fotmob)
)

resolved_id = pd.to_numeric(
    df["Resolved_Player_ID"],
    errors="coerce",
).astype("Int64")

df["Player_Key"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="string",
)

has_fotmob = (
    resolved_id.notna()
    & resolved_id.gt(0)
)

df.loc[
    has_fotmob,
    "Player_Key",
] = (
    "FM_"
    + resolved_id.loc[
        has_fotmob
    ].astype("string")
)

need_opta = (
    df["Player_Key"].isna()
    & df["Opta_ID"].notna()
)

df.loc[
    need_opta,
    "Player_Key",
] = (
    "OPTA_"
    + df.loc[
        need_opta,
        "Opta_ID",
    ].astype("string")
)

assert (
    df["Player_Key"].notna().all()
), "Some match rows have no player key."


# ---------- Match-level performance fields ----------
df["Appearance"] = (
    df["Minutes"] > 0
).astype(int)

df["Start_Num"] = (
    df["Starts"]
    .astype("string")
    .str.lower()
    .isin(["true", "1", "yes"])
    .astype(int)
)

df["Rated_Match"] = (
    df["Rating"].notna()
).astype(int)

df["Rated_Minutes"] = np.where(
    df["Rating"].notna(),
    df["Minutes"],
    0,
)

df["Rating_x_Minutes"] = np.where(
    df["Rating"].notna(),
    df["Rating"] * df["Minutes"],
    0,
)


# ---------- Aggregate to player × season ----------
player_season = (
    df
    .groupby(
        ["Season", "Player_Key"],
        as_index=False,
    )
    .agg(
        Minutes=("Minutes", "sum"),
        Appearances=("Appearance", "sum"),
        Starts=("Start_Num", "sum"),
        Rated_Matches=("Rated_Match", "sum"),
        Rated_Minutes=("Rated_Minutes", "sum"),
        Rating_Weighted_Sum=("Rating_x_Minutes", "sum"),
        Season_Rating_Exact=("Rating", "mean"),
    )
)

player_season["Season_Rating_Minutes_Weighted"] = np.where(
    player_season["Rated_Minutes"] > 0,
    (
        player_season["Rating_Weighted_Sum"]
        / player_season["Rated_Minutes"]
    ),
    np.nan,
)

player_season["Rating_Match_Coverage_Pct"] = np.where(
    player_season["Appearances"] > 0,
    (
        player_season["Rated_Matches"]
        / player_season["Appearances"]
        * 100
    ),
    np.nan,
)

player_season["Rating_Minute_Coverage_Pct"] = np.where(
    player_season["Minutes"] > 0,
    (
        player_season["Rated_Minutes"]
        / player_season["Minutes"]
        * 100
    ),
    np.nan,
)


# ---------- Primary team in each season ----------
team_minutes = (
    df
    .groupby(
        ["Season", "Player_Key", "Team"],
        as_index=False,
    )["Minutes"]
    .sum()
)

primary_team = (
    team_minutes
    .sort_values(
        ["Season", "Player_Key", "Minutes"],
        ascending=[True, True, False],
    )
    .drop_duplicates(
        ["Season", "Player_Key"]
    )
    [
        [
            "Season",
            "Player_Key",
            "Team",
        ]
    ]
)

player_season = player_season.merge(
    primary_team,
    on=["Season", "Player_Key"],
    how="left",
    validate="one_to_one",
)


# ---------- Attach profile role ----------
role_lookup = (
    roles[
        [
            "Player_Key",
            "Player",
            "Final_Position",
            "Analysis_Role",
        ]
    ]
    .drop_duplicates("Player_Key")
)

player_season = (
    player_season
    .merge(
        role_lookup,
        on="Player_Key",
        how="left",
        validate="many_to_one",
    )
    .rename(
        columns={
            "Analysis_Role": "Role",
        }
    )
)


# ---------- Eligibility ----------
MIN_MINUTES = 900

eligible_performance = (
    player_season.loc[
        (player_season["Minutes"] >= MIN_MINUTES)
        & player_season["Role"].notna()
        & player_season["Season_Rating_Exact"].notna()
    ]
    .copy()
)


# ---------- Seven audited season-role corrections ----------
role_overrides = pd.DataFrame([
    ("2023/2024", "FM_493165", "Attacking Midfielder"),
    ("2024/2025", "FM_493165", "Attacking Midfielder"),
    ("2023/2024", "FM_688278", "Full-back / Wing-back"),
    ("2023/2024", "FM_956621", "Full-back / Wing-back"),
    ("2024/2025", "FM_935379", "Attacking Midfielder"),
    ("2025/2026", "FM_1067763", "Full-back / Wing-back"),
    ("2025/2026", "FM_523193", "Central / Defensive Midfielder"),
], columns=[
    "Season",
    "Player_Key",
    "Corrected_Role",
])

eligible_performance = (
    eligible_performance
    .merge(
        role_overrides,
        on=["Season", "Player_Key"],
        how="left",
        validate="one_to_one",
    )
)

eligible_performance["Original_Role"] = (
    eligible_performance["Role"]
)

eligible_performance["Role"] = (
    eligible_performance["Corrected_Role"]
    .fillna(
        eligible_performance["Role"]
    )
)

eligible_performance["Role_Corrected"] = (
    eligible_performance["Corrected_Role"]
    .notna()
)

eligible_performance["Season_Rating"] = (
    eligible_performance["Season_Rating_Exact"]
    .round(3)
)


print("=" * 76)
print("Q2 PERFORMANCE UNIVERSE")
print("=" * 76)

print(
    "Match-level rows:",
    f"{len(df):,}",
)

print(
    "Player-season rows:",
    f"{len(player_season):,}",
)

print(
    "Eligible player-seasons:",
    f"{len(eligible_performance):,}",
)

print(
    "Role corrections:",
    f"{eligible_performance['Role_Corrected'].sum():,}",
)

display(
    eligible_performance[
        [
            "Season",
            "Player",
            "Team",
            "Role",
            "Minutes",
            "Appearances",
            "Season_Rating",
            "Season_Rating_Minutes_Weighted",
        ]
    ]
    .sort_values(
        ["Season", "Season_Rating"],
        ascending=[True, False],
    )
    .head(20)
)

In [ ]:
# ============================================================
# Q2.3 — RANK PLAYERS WITHIN SEASON × ROLE
# ============================================================

rating_rank = eligible_performance.copy()

rating_rank["Rating_Rank"] = (
    rating_rank
    .groupby(
        ["Season", "Role"]
    )["Season_Rating_Exact"]
    .rank(
        method="min",
        ascending=False,
    )
    .astype("Int64")
)

rating_rank["Role_Group_Size"] = (
    rating_rank
    .groupby(
        ["Season", "Role"]
    )["Player_Key"]
    .transform("size")
)


def percentile_0_100(series):
    n = series.notna().sum()

    if n < 2:
        return pd.Series(
            np.nan,
            index=series.index,
        )

    ascending_rank = series.rank(
        method="average",
        ascending=True,
        na_option="keep",
    )

    return (
        (ascending_rank - 1)
        / (n - 1)
        * 100
    )


rating_rank["Rating_Percentile"] = (
    rating_rank
    .groupby(
        ["Season", "Role"]
    )["Season_Rating_Exact"]
    .transform(
        percentile_0_100
    )
    .round(1)
)

rating_rank["Rating_Rank_Label"] = (
    rating_rank["Rating_Rank"].astype("string")
    + " / "
    + rating_rank["Role_Group_Size"].astype("string")
)


# ---------- Robustness check ----------
rating_rank["Abs_Rating_Difference"] = (
    rating_rank["Season_Rating_Exact"]
    - rating_rank["Season_Rating_Minutes_Weighted"]
).abs()

rating_spearman = (
    rating_rank[
        [
            "Season_Rating_Exact",
            "Season_Rating_Minutes_Weighted",
        ]
    ]
    .corr(method="spearman")
    .iloc[0, 1]
)

print(
    "Mean vs minutes-weighted rating Spearman correlation:",
    round(rating_spearman, 4),
)

print(
    "Median absolute rating difference:",
    round(
        rating_rank["Abs_Rating_Difference"].median(),
        4,
    ),
)


top_5_by_role = (
    rating_rank.loc[
        rating_rank["Rating_Rank"] <= 5,
        [
            "Season",
            "Role",
            "Rating_Rank",
            "Role_Group_Size",
            "Player",
            "Team",
            "Season_Rating",
            "Rating_Percentile",
        ],
    ]
    .sort_values(
        ["Season", "Role", "Rating_Rank", "Player"]
    )
)

display(top_5_by_role.head(50))

rating_rank.to_csv(
    TABLE_DIR / "q2_player_season_performance_rank.csv",
    index=False,
)

top_5_by_role.to_csv(
    TABLE_DIR / "q2_top5_by_season_role.csv",
    index=False,
)

In [ ]:
# ============================================================
# Q2.4 — FOTMOB RATING DISTRIBUTION BY ROLE
# ============================================================

plot_data = [
    rating_rank.loc[
        rating_rank["Role"] == role,
        "Season_Rating_Exact",
    ]
    .dropna()
    .values
    for role in role_order
]

role_counts = [
    rating_rank.loc[
        rating_rank["Role"] == role
    ].shape[0]
    for role in role_order
]

fig, ax = plt.subplots(
    figsize=(12, 8),
    dpi=150,
)

ax.boxplot(
    plot_data,
    tick_labels=[
        f"{role}\n(n={n})"
        for role, n in zip(
            role_order,
            role_counts,
        )
    ],
    showfliers=False,
)

ax.set_title(
    "Distribution of FotMob Season Rating by Football Role",
    fontsize=16,
    fontweight="bold",
)
ax.set_xlabel("Analysis role")
ax.set_ylabel("Mean FotMob match rating")
ax.tick_params(axis="x", rotation=28)
ax.grid(axis="y", alpha=0.20)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "q2_fotmob_rating_by_role.png",
    dpi=500,
    bbox_inches="tight",
)
plt.show()

# Q3 — Does Football Performance Relate to eBay Market Presence?

In [ ]:
# ============================================================
# Q3 — PLAYER PERFORMANCE × EBAY PRESENCE
# ============================================================

rating_rank["FotMob_Player_ID"] = pd.to_numeric(
    rating_rank["Player_Key"]
    .str.replace(
        "FM_",
        "",
        regex=False,
    ),
    errors="coerce",
).astype("Int64")

season_order = {
    "2023/2024": 1,
    "2024/2025": 2,
    "2025/2026": 3,
}

rating_rank["Season_Order"] = (
    rating_rank["Season"]
    .map(season_order)
)

latest_performance = (
    rating_rank
    .sort_values(
        ["Player_Key", "Season_Order"]
    )
    .groupby(
        "Player_Key",
        as_index=False,
    )
    .tail(1)
    [
        [
            "Player_Key",
            "FotMob_Player_ID",
            "Player",
            "Team",
            "Season",
            "Role",
            "Minutes",
            "Season_Rating",
            "Rating_Rank",
            "Role_Group_Size",
            "Rating_Percentile",
        ]
    ]
    .rename(
        columns={
            "Season": "Latest_Eligible_Season",
            "Team": "Latest_Team",
            "Role": "Latest_Role",
            "Minutes": "Latest_Minutes",
            "Season_Rating": "Latest_Mean_FotMob_Rating",
            "Rating_Rank": "Latest_Rating_Rank",
            "Role_Group_Size": "Latest_Role_Group_Size",
            "Rating_Percentile": "Latest_Rating_Percentile",
        }
    )
    .reset_index(drop=True)
)

mean_performance = (
    rating_rank
    .groupby(
        "Player_Key",
        as_index=False,
    )
    .agg(
        Mean_Rating_Percentile=(
            "Rating_Percentile",
            "mean",
        ),
        Eligible_Seasons=(
            "Season",
            "nunique",
        ),
    )
)

player_market = latest_performance.merge(
    mean_performance,
    on="Player_Key",
    how="left",
    validate="one_to_one",
)

ebay_player_ids = set(
    ebay["FotMob_Player_ID"]
    .dropna()
    .astype(int)
    .unique()
)

player_market["Has_eBay_Listing"] = (
    player_market["FotMob_Player_ID"]
    .astype("Int64")
    .isin(ebay_player_ids)
)

player_market["eBay_Presence"] = np.where(
    player_market["Has_eBay_Listing"],
    "Represented on eBay",
    "Not represented on eBay",
)

performance_band_order = [
    "Bottom 25%",
    "25–50%",
    "50–75%",
    "Top 25%",
]

player_market["Performance_Band"] = pd.cut(
    player_market["Latest_Rating_Percentile"],
    bins=[
        -0.001,
        25,
        50,
        75,
        100,
    ],
    labels=performance_band_order,
    include_lowest=True,
)

presence_by_band = (
    player_market
    .groupby(
        "Performance_Band",
        observed=False,
    )
    .agg(
        Players=("FotMob_Player_ID", "size"),
        Players_on_eBay=("Has_eBay_Listing", "sum"),
    )
    .reset_index()
)

presence_by_band["Presence_Rate_Pct"] = (
    presence_by_band["Players_on_eBay"]
    / presence_by_band["Players"]
    * 100
)

display(
    presence_by_band.round(2)
)


represented_perf = player_market.loc[
    player_market["Has_eBay_Listing"],
    "Latest_Rating_Percentile",
].dropna()

not_represented_perf = player_market.loc[
    ~player_market["Has_eBay_Listing"],
    "Latest_Rating_Percentile",
].dropna()

presence_u, presence_p = mannwhitneyu(
    represented_perf,
    not_represented_perf,
    alternative="two-sided",
)

print(
    "Mann–Whitney p-value "
    "(represented vs not represented):",
    f"{presence_p:.4g}",
)

player_market.to_csv(
    TABLE_DIR / "q3_player_market_presence.csv",
    index=False,
)

presence_by_band.to_csv(
    TABLE_DIR / "q3_ebay_presence_by_performance_band.csv",
    index=False,
)


fig, ax = plt.subplots(
    figsize=(9, 6),
    dpi=150,
)

bars = ax.bar(
    presence_by_band["Performance_Band"].astype(str),
    presence_by_band["Presence_Rate_Pct"],
)

for bar, rate, n in zip(
    bars,
    presence_by_band["Presence_Rate_Pct"],
    presence_by_band["Players"],
):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{rate:.1f}%\n(n={n})",
        ha="center",
        va="bottom",
        fontsize=9,
    )

ax.set_title(
    "eBay Market Presence by Football Performance Band",
    fontsize=15,
    fontweight="bold",
)
ax.set_xlabel("Latest eligible performance percentile")
ax.set_ylabel("Players represented on eBay (%)")
ax.set_ylim(
    0,
    min(
        100,
        presence_by_band["Presence_Rate_Pct"].max() + 15,
    ),
)
ax.grid(axis="y", alpha=0.20)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "q3_ebay_presence_by_performance_band.png",
    dpi=500,
    bbox_inches="tight",
)
plt.show()

# Q4 — Does Football Performance Relate to Market Attention or Asking Price?

In [ ]:
# ============================================================
# Q4.1 — PERFORMANCE vs LISTING VOLUME AND PRICE
# ============================================================

market_by_player = (
    ebay
    .groupby(
        "FotMob_Player_ID",
        as_index=False,
    )
    .agg(
        Observed_Listings=(
            "source_row_id",
            "nunique",
        ),
        Median_Asking_Price_USD=(
            "asking_price_usd",
            "median",
        ),
        Mean_Asking_Price_USD=(
            "asking_price_usd",
            "mean",
        ),
    )
)

market_performance = player_market.merge(
    market_by_player,
    on="FotMob_Player_ID",
    how="left",
    validate="one_to_one",
)

represented = (
    market_performance.loc[
        market_performance["Has_eBay_Listing"]
    ]
    .dropna(
        subset=[
            "Observed_Listings",
            "Median_Asking_Price_USD",
        ]
    )
    .copy()
)

listing_rho, listing_p = spearmanr(
    represented["Latest_Rating_Percentile"],
    represented["Observed_Listings"],
)

price_rho, price_p = spearmanr(
    represented["Latest_Rating_Percentile"],
    represented["Median_Asking_Price_USD"],
)

q4_overall = pd.DataFrame([
    {
        "Relationship":
            "Performance percentile vs listing count",
        "Players":
            len(represented),
        "Spearman_Rho":
            listing_rho,
        "P_Value":
            listing_p,
    },
    {
        "Relationship":
            "Performance percentile vs median asking price",
        "Players":
            len(represented),
        "Spearman_Rho":
            price_rho,
        "P_Value":
            price_p,
    },
])

display(q4_overall.round(4))

q4_overall.to_csv(
    TABLE_DIR / "q4_overall_performance_market_relationships.csv",
    index=False,
)


fig, ax = plt.subplots(
    figsize=(9, 6),
    dpi=150,
)

ax.scatter(
    represented["Latest_Rating_Percentile"],
    represented["Observed_Listings"],
    alpha=0.45,
    s=35,
)

ax.set_yscale("log")

ax.set_title(
    "Football Performance vs Observed eBay Listing Volume",
    fontsize=15,
    fontweight="bold",
)
ax.set_xlabel("Latest role-relative FotMob performance percentile")
ax.set_ylabel("Observed listings per player (log scale)")
ax.grid(alpha=0.20)

ax.text(
    0.03,
    0.96,
    (
        f"Spearman ρ = {listing_rho:.3f}\n"
        f"p = {listing_p:.3g}\n"
        f"n = {len(represented)}"
    ),
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=9.5,
    bbox=dict(
        boxstyle="round",
        facecolor="white",
        alpha=0.80,
        edgecolor="gray",
    ),
)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "q4_performance_vs_listing_volume.png",
    dpi=500,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# ============================================================
# Q4.2 — PERFORMANCE vs PRICE WITHIN BRAND × PRODUCT LINE
# ============================================================

performance_players = (
    player_market.loc[
        player_market["Has_eBay_Listing"],
        [
            "FotMob_Player_ID",
            "Player",
            "Latest_Role",
            "Latest_Rating_Percentile",
        ],
    ]
    .copy()
)

controlled = (
    ebay.loc[
        ebay["FotMob_Player_ID"].notna()
        & ebay["asking_price_usd"].notna()
        & ebay["product_line"].notna()
        & ebay["brand_clean"].notna()
    ]
    .merge(
        performance_players,
        on="FotMob_Player_ID",
        how="inner",
        validate="many_to_one",
    )
)

player_line = (
    controlled
    .groupby(
        [
            "brand_clean",
            "product_line",
            "FotMob_Player_ID",
        ],
        as_index=False,
    )
    .agg(
        Player=("Player", "first"),
        Latest_Role=("Latest_Role", "first"),
        Performance_Percentile=(
            "Latest_Rating_Percentile",
            "first",
        ),
        Observed_Listings=(
            "source_row_id",
            "nunique",
        ),
        Median_Asking_Price_USD=(
            "asking_price_usd",
            "median",
        ),
    )
)

line_coverage = (
    player_line
    .groupby(
        ["brand_clean", "product_line"],
        as_index=False,
    )
    .agg(
        Players=("FotMob_Player_ID", "nunique"),
        Listings=("Observed_Listings", "sum"),
    )
    .sort_values(
        ["Players", "Listings"],
        ascending=False,
    )
)

MIN_PLAYERS_PER_LINE = 50

major_lines = line_coverage.loc[
    line_coverage["Players"] >= MIN_PLAYERS_PER_LINE,
    ["brand_clean", "product_line"],
]

player_line_major = player_line.merge(
    major_lines,
    on=["brand_clean", "product_line"],
    how="inner",
    validate="many_to_one",
)


within_line_rows = []

for (brand, product_line), group in player_line_major.groupby(
    ["brand_clean", "product_line"]
):
    rho, p_value = spearmanr(
        group["Performance_Percentile"],
        group["Median_Asking_Price_USD"],
    )

    within_line_rows.append({
        "Brand": brand,
        "Product_Line": product_line,
        "Players": len(group),
        "Spearman_Rho": rho,
        "P_Value": p_value,
    })

within_line_results = (
    pd.DataFrame(within_line_rows)
    .sort_values("Players", ascending=False)
    .reset_index(drop=True)
)

display(
    within_line_results.round(4)
)


# Price percentile inside each Brand × Product Line.
player_line_major["Price_Percentile_Within_Line"] = (
    player_line_major
    .groupby(
        ["brand_clean", "product_line"]
    )["Median_Asking_Price_USD"]
    .rank(
        method="average",
        pct=True,
    )
    * 100
)

player_adjusted_price = (
    player_line_major
    .groupby(
        "FotMob_Player_ID",
        as_index=False,
    )
    .agg(
        Player=("Player", "first"),
        Latest_Role=("Latest_Role", "first"),
        Performance_Percentile=(
            "Performance_Percentile",
            "first",
        ),
        Product_Lines=("product_line", "nunique"),
        Adjusted_Price_Percentile=(
            "Price_Percentile_Within_Line",
            "median",
        ),
    )
)

adjusted_rho, adjusted_p = spearmanr(
    player_adjusted_price["Performance_Percentile"],
    player_adjusted_price["Adjusted_Price_Percentile"],
)

print(
    f"Product-line-adjusted relationship: "
    f"rho = {adjusted_rho:.3f}, "
    f"p = {adjusted_p:.4g}, "
    f"n = {len(player_adjusted_price)} players"
)

within_line_results.to_csv(
    TABLE_DIR / "q4_within_product_line_correlations.csv",
    index=False,
)

player_adjusted_price.to_csv(
    TABLE_DIR / "q4_player_adjusted_price_percentile.csv",
    index=False,
)

In [ ]:
# ============================================================
# Q4.3 — STRICT LIKE-FOR-LIKE CARD SAMPLE
#
# 2023 Panini Prizm
# non-auto • non-patch • non-rookie
# ungraded • unnumbered
# ============================================================

controlled_strict = ebay.merge(
    performance_players,
    on="FotMob_Player_ID",
    how="inner",
    validate="many_to_one",
    suffixes=("", "_Performance"),
)

controlled_strict["Numbered"] = (
    controlled_strict["serial"].notna()
)

comparable_cards = controlled_strict.loc[
    (controlled_strict["brand_clean"] == "Panini")
    & (controlled_strict["product_line"] == "Prizm")
    & (controlled_strict["year"] == 2023)
    & (controlled_strict["autograph"] == False)
    & (controlled_strict["patch"] == False)
    & (controlled_strict["rookie"] == False)
    & (controlled_strict["graded"] == False)
    & (controlled_strict["Numbered"] == False)
    & controlled_strict["asking_price_usd"].notna()
].copy()

q43 = (
    comparable_cards
    .groupby(
        "FotMob_Player_ID",
        as_index=False,
    )
    .agg(
        Player=("Player", "first"),
        Latest_Role=("Latest_Role", "first"),
        Performance_Percentile=(
            "Latest_Rating_Percentile",
            "first",
        ),
        Observed_Listings=(
            "source_row_id",
            "nunique",
        ),
        Median_Asking_Price_USD=(
            "asking_price_usd",
            "median",
        ),
        Mean_Asking_Price_USD=(
            "asking_price_usd",
            "mean",
        ),
        Min_Asking_Price_USD=(
            "asking_price_usd",
            "min",
        ),
        Max_Asking_Price_USD=(
            "asking_price_usd",
            "max",
        ),
    )
)

strict_rho, strict_p = spearmanr(
    q43["Performance_Percentile"],
    q43["Median_Asking_Price_USD"],
)

print("=" * 78)
print("STRICT COMPARABLE-CARD SAMPLE")
print("=" * 78)
print("Listings:", len(comparable_cards))
print("Players :", len(q43))
print(
    f"Spearman rho = {strict_rho:.3f}, "
    f"p = {strict_p:.4g}"
)


band_order = [
    "0–25th percentile",
    "25–50th percentile",
    "50–75th percentile",
    "75–100th percentile",
]

q43["Performance_Band"] = pd.cut(
    q43["Performance_Percentile"],
    bins=[
        -0.001,
        25,
        50,
        75,
        100,
    ],
    labels=band_order,
    include_lowest=True,
)

price_by_band = (
    q43
    .groupby(
        "Performance_Band",
        observed=False,
    )
    .agg(
        Players=("FotMob_Player_ID", "size"),
        Median_Player_Price_USD=(
            "Median_Asking_Price_USD",
            "median",
        ),
        Mean_Player_Price_USD=(
            "Median_Asking_Price_USD",
            "mean",
        ),
    )
    .reset_index()
)

display(price_by_band.round(2))

q43.to_csv(
    TABLE_DIR / "q43_comparable_prizm_player_price.csv",
    index=False,
)

price_by_band.to_csv(
    TABLE_DIR / "q43_comparable_prizm_price_by_band.csv",
    index=False,
)


plot_data = [
    q43.loc[
        q43["Performance_Band"] == band,
        "Median_Asking_Price_USD",
    ]
    .dropna()
    .values
    for band in band_order
]

band_n = [
    q43.loc[
        q43["Performance_Band"] == band
    ].shape[0]
    for band in band_order
]

fig, ax = plt.subplots(
    figsize=(10, 6.5),
    dpi=150,
)

ax.boxplot(
    plot_data,
    tick_labels=[
        f"{band}\n(n={n})"
        for band, n in zip(
            band_order,
            band_n,
        )
    ],
    showfliers=False,
)

ax.set_yscale("log")

ax.set_title(
    "Football Performance and Asking Price Within Comparable Cards",
    fontsize=15,
    fontweight="bold",
)
ax.text(
    0.5,
    1.01,
    (
        "2023 Panini Prizm • non-auto • non-patch • "
        "non-rookie • ungraded • unnumbered"
    ),
    transform=ax.transAxes,
    ha="center",
    fontsize=9,
)

ax.set_xlabel("Football performance percentile band")
ax.set_ylabel("Player median asking price (USD, log scale)")
ax.grid(axis="y", alpha=0.20)

ax.text(
    0.03,
    0.96,
    (
        f"Spearman ρ = {strict_rho:.3f}\n"
        f"p = {strict_p:.3g}\n"
        f"n = {len(q43)} players"
    ),
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=9,
    bbox=dict(
        boxstyle="round",
        facecolor="white",
        alpha=0.80,
        edgecolor="gray",
    ),
)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "q43_comparable_cards_performance_vs_price.png",
    dpi=500,
    bbox_inches="tight",
)
plt.show()

# Q5 — Where Does the Card Market Diverge from Football Performance?

This section standardizes performance within `Season × Role`, standardizes log card price within football role, and highlights large mismatches in the strict comparable-card sample.

In [ ]:
# ============================================================
# Q5 — MAGNITUDE-SENSITIVE PERFORMANCE × PRICE MISMATCH
# ============================================================

# ---------- Standardize performance within Season × Role ----------
performance_z = rating_rank.copy()

performance_z["Role_Rating_Mean"] = (
    performance_z
    .groupby(
        ["Season", "Role"]
    )["Season_Rating_Exact"]
    .transform("mean")
)

performance_z["Role_Rating_SD"] = (
    performance_z
    .groupby(
        ["Season", "Role"]
    )["Season_Rating_Exact"]
    .transform("std")
)

performance_z["Performance_Z"] = (
    (
        performance_z["Season_Rating_Exact"]
        - performance_z["Role_Rating_Mean"]
    )
    / performance_z["Role_Rating_SD"]
)

latest_z = (
    performance_z
    .sort_values(
        ["Player_Key", "Season_Order"]
    )
    .groupby(
        "Player_Key",
        as_index=False,
    )
    .tail(1)
    [
        [
            "FotMob_Player_ID",
            "Season",
            "Role",
            "Season_Rating_Exact",
            "Role_Rating_Mean",
            "Role_Rating_SD",
            "Performance_Z",
        ]
    ]
    .copy()
)

q43_mag = q43.merge(
    latest_z,
    on="FotMob_Player_ID",
    how="inner",
    validate="one_to_one",
)

q43_mag["Log_Median_Price"] = np.log(
    q43_mag["Median_Asking_Price_USD"]
)


# ---------- Magnitude-sensitive relationship ----------
pearson_r, pearson_p = pearsonr(
    q43_mag["Performance_Z"],
    q43_mag["Log_Median_Price"],
)

model = smf.ols(
    formula=(
        "Log_Median_Price "
        "~ Performance_Z "
        "+ C(Role)"
    ),
    data=q43_mag,
).fit()

beta = model.params["Performance_Z"]
beta_p = model.pvalues["Performance_Z"]

price_change_pct = (
    np.exp(beta) - 1
) * 100

print("=" * 78)
print("MAGNITUDE-SENSITIVE PERFORMANCE × PRICE")
print("=" * 78)

print(
    f"Pearson r = {pearson_r:.3f}, "
    f"p = {pearson_p:.4g}"
)

print(
    f"Role-adjusted performance coefficient = "
    f"{beta:.4f}"
)

print(
    f"Regression p-value = {beta_p:.4g}"
)

print(
    f"Estimated price change for +1 SD performance = "
    f"{price_change_pct:.1f}%"
)

print(
    f"Model R-squared = {model.rsquared:.3f}"
)


# ---------- Price Z-score within role ----------
q5 = q43_mag.copy()

q5["Role_N"] = (
    q5
    .groupby("Role")["FotMob_Player_ID"]
    .transform("size")
)

q5["Role_LogPrice_Mean"] = (
    q5
    .groupby("Role")["Log_Median_Price"]
    .transform("mean")
)

q5["Role_LogPrice_SD"] = (
    q5
    .groupby("Role")["Log_Median_Price"]
    .transform("std")
)

q5["Price_Z_Within_Role"] = (
    (
        q5["Log_Median_Price"]
        - q5["Role_LogPrice_Mean"]
    )
    / q5["Role_LogPrice_SD"]
)

q5["Mismatch_Score"] = (
    q5["Performance_Z"]
    - q5["Price_Z_Within_Role"]
)

PERFORMANCE_THRESHOLD = 0.50
PRICE_THRESHOLD = 0.50

conditions = [
    (
        (q5["Performance_Z"] >= PERFORMANCE_THRESHOLD)
        & (q5["Price_Z_Within_Role"] <= -PRICE_THRESHOLD)
    ),
    (
        (q5["Performance_Z"] <= -PERFORMANCE_THRESHOLD)
        & (q5["Price_Z_Within_Role"] >= PRICE_THRESHOLD)
    ),
]

choices = [
    "Performance-rich / lower-priced",
    "Market-premium / lower-performance",
]

q5["Mismatch_Type"] = np.select(
    conditions,
    choices,
    default="Other",
)

q5_stable = q5.loc[
    (q5["Observed_Listings"] >= 3)
    & (q5["Role_N"] >= 20)
].copy()

performance_rich = (
    q5_stable.loc[
        q5_stable["Mismatch_Type"]
        == "Performance-rich / lower-priced"
    ]
    .sort_values(
        "Mismatch_Score",
        ascending=False,
    )
)

market_premium = (
    q5_stable.loc[
        q5_stable["Mismatch_Type"]
        == "Market-premium / lower-performance"
    ]
    .sort_values(
        "Mismatch_Score",
        ascending=True,
    )
)

print("\nStable mismatch sample:")
print(
    q5_stable["Mismatch_Type"]
    .value_counts()
)

display(
    performance_rich[
        [
            "Player",
            "Role",
            "Observed_Listings",
            "Season_Rating_Exact",
            "Performance_Z",
            "Median_Asking_Price_USD",
            "Price_Z_Within_Role",
            "Mismatch_Score",
        ]
    ]
    .head(15)
    .round(3)
)

display(
    market_premium[
        [
            "Player",
            "Role",
            "Observed_Listings",
            "Season_Rating_Exact",
            "Performance_Z",
            "Median_Asking_Price_USD",
            "Price_Z_Within_Role",
            "Mismatch_Score",
        ]
    ]
    .head(15)
    .round(3)
)

q43_mag.to_csv(
    TABLE_DIR / "q5_magnitude_sensitive_performance_price.csv",
    index=False,
)

q5.to_csv(
    TABLE_DIR / "q5_performance_price_mismatch_all.csv",
    index=False,
)

q5_stable.to_csv(
    TABLE_DIR / "q5_performance_price_mismatch_stable.csv",
    index=False,
)


# ---------- Mismatch map ----------
fig, ax = plt.subplots(
    figsize=(10, 8),
    dpi=150,
)

ax.scatter(
    q5["Performance_Z"],
    q5["Price_Z_Within_Role"],
    alpha=0.30,
    s=35,
    label="Other players",
)

under_plot = q5_stable.loc[
    q5_stable["Mismatch_Type"]
    == "Performance-rich / lower-priced"
]

premium_plot = q5_stable.loc[
    q5_stable["Mismatch_Type"]
    == "Market-premium / lower-performance"
]

ax.scatter(
    under_plot["Performance_Z"],
    under_plot["Price_Z_Within_Role"],
    s=60,
    alpha=0.85,
    label="Performance-rich / lower-priced",
)

ax.scatter(
    premium_plot["Performance_Z"],
    premium_plot["Price_Z_Within_Role"],
    s=60,
    alpha=0.85,
    label="Market-premium / lower-performance",
)

ax.axvline(
    0,
    linestyle="--",
    linewidth=1,
    alpha=0.55,
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1,
    alpha=0.55,
)

lim = max(
    abs(q5["Performance_Z"]).max(),
    abs(q5["Price_Z_Within_Role"]).max(),
)
lim = np.ceil(lim * 10) / 10

ax.plot(
    [-lim, lim],
    [-lim, lim],
    linestyle=":",
    linewidth=1.5,
    alpha=0.65,
    label="Equal relative standing",
)

for _, row in performance_rich.head(5).iterrows():
    ax.annotate(
        row["Player"],
        (
            row["Performance_Z"],
            row["Price_Z_Within_Role"],
        ),
        xytext=(6, -10),
        textcoords="offset points",
        fontsize=8,
    )

for _, row in market_premium.head(5).iterrows():
    ax.annotate(
        row["Player"],
        (
            row["Performance_Z"],
            row["Price_Z_Within_Role"],
        ),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=8,
    )

ax.set_title(
    "Where Does the Card Market Diverge from Football Performance?",
    fontsize=15,
    fontweight="bold",
)
ax.text(
    0.5,
    1.01,
    (
        "2023 Panini Prizm • non-auto • non-patch • "
        "non-rookie • ungraded • unnumbered"
    ),
    transform=ax.transAxes,
    ha="center",
    fontsize=9,
)

ax.set_xlabel(
    "Standardized football performance within Season × Role"
)
ax.set_ylabel(
    "Standardized log asking price within football role"
)

ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.grid(alpha=0.18)
ax.legend(
    loc="lower left",
    fontsize=8,
)

plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "q5_performance_price_mismatch_map.png",
    dpi=500,
    bbox_inches="tight",
)
plt.show()

# Export All Results

The notebook has already saved figures and CSV tables while running.  
The last cell creates one ZIP file containing the complete output folder.

Set `DOWNLOAD_RESULTS = True` if you want Colab to automatically download the ZIP.

In [ ]:
# ============================================================
# EXPORT OUTPUTS
# ============================================================

# Save a compact run summary.
run_summary = pd.DataFrame({
    "Metric": [
        "eBay listings",
        "Unique eBay players",
        "FotMob match rows",
        "Player profiles",
        "Eligible player-seasons",
        "Performance-eligible unique players",
        "Players represented on eBay",
        "Strict comparable-card players",
    ],
    "Value": [
        len(ebay),
        ebay["FotMob_Player_ID"].nunique(),
        len(matches),
        len(profiles),
        len(rating_rank),
        rating_rank["FotMob_Player_ID"].nunique(),
        player_market["Has_eBay_Listing"].sum(),
        len(q43),
    ],
})

run_summary.to_csv(
    TABLE_DIR / "run_summary.csv",
    index=False,
)

display(run_summary)

zip_base = Path("football_card_analysis_output")
zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=OUTPUT_DIR,
)

print("Created:", zip_path)

DOWNLOAD_RESULTS = False

if DOWNLOAD_RESULTS and "google.colab" in sys.modules:
    from google.colab import files
    files.download(zip_path)